In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [ ]:
from torchvision import transforms

transform = transforms.Compose([

    # tamaño de EfficientNet
    transforms.Resize((224,224)),

    # espejo horizontal
    transforms.RandomHorizontalFlip(p=0.5),

    # rotaciones más agresivas
    transforms.RandomRotation(25),

    # cambios de iluminación y color
    transforms.ColorJitter(
        brightness=0.4,
        contrast=0.4,
        saturation=0.4,
        hue=0.1
    ),

    # zoom + desplazamiento
    transforms.RandomAffine(
        degrees=0,
        translate=(0.15, 0.15),
        scale=(0.8, 1.2)
    ),

    # perspectiva tipo cámara real
    transforms.RandomPerspective(
        distortion_scale=0.25,
        p=0.4
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    ),

    # oclusiones parciales (mano, sombras, objetos)
    transforms.RandomErasing(
        p=0.25,
        scale=(0.02,0.12)
    )
])

In [ ]:
import os
import json
from torchvision import datasets
from torch.utils.data import DataLoader

DATASET_PATH = "../data/food-101/images"

# =========================
# CARGAR DATASET
# =========================
dataset = datasets.ImageFolder(
    root=DATASET_PATH,
    transform=transform
)

# clases reales detectadas automáticamente
CLASSES = dataset.classes

# guardar clases para la app
os.makedirs("../models", exist_ok=True)

with open("../models/classes.json", "w") as f:
    json.dump(CLASSES, f)

# =========================
# DATALOADER
# =========================
loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print("Total imágenes:", len(dataset))
print("Total clases:", len(CLASSES))
print("Clases reales:", CLASSES)

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0

OLD_MODEL_PATH = "../models/modelo_efficientnet.pth"

# =========================
# DETECTAR CLASES DEL MODELO ANTERIOR
# =========================
checkpoint = torch.load(
    OLD_MODEL_PATH,
    map_location="cpu"
)

old_classes = checkpoint[
    "classifier.1.weight"
].shape[0]

# =========================
# MODELO NUEVO (N clases)
# =========================
model = efficientnet_b0(weights=None)

new_num_classes = len(CLASSES)

# nueva capa final
new_classifier = nn.Linear(
    1280,
    new_num_classes
)

# =========================
# CARGAR MODELO ANTERIOR
# =========================
old_model = efficientnet_b0(weights=None)

old_model.classifier[1] = nn.Linear(
    1280,
    old_classes
)

old_model.load_state_dict(checkpoint)

# =========================
# COPIAR CONOCIMIENTO VIEJO
# =========================
with torch.no_grad():

    # inicializar nuevas neuronas
    nn.init.xavier_uniform_(
        new_classifier.weight
    )

    # copiar clases antiguas
    new_classifier.weight[
        :old_classes
    ] = old_model.classifier[1].weight

    new_classifier.bias[
        :old_classes
    ] = old_model.classifier[1].bias


# reemplazar classifier
model.classifier[1] = new_classifier

# copiar backbone completo
model.features.load_state_dict(
    old_model.features.state_dict()
)

# =========================
# FASE 1
# congelar backbone
# =========================
for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True


print("Modelo incremental listo")
print("Clases antiguas:", old_classes)
print("Clases nuevas:", len(CLASSES))

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# =========================
# DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Usando:", device)

criterion = nn.CrossEntropyLoss()


# =========================
# FASE 1
# SOLO CLASIFICADOR
# =========================
print("Fase 1: Adaptando nuevas clases")

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

optimizer = optim.Adam(
    model.classifier.parameters(),
    lr=0.001
)

EPOCHS = 8

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for images, labels in loader:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"[F1] Epoch {epoch+1}/{EPOCHS}, "
        f"Loss: {total_loss:.4f}"
    )


# =========================
# FASE 2
# FINE TUNING SUAVE
# =========================
print("Fase 2: Fine-tuning global")

for param in model.parameters():
    param.requires_grad = True

optimizer = optim.Adam(
    model.parameters(),
    lr=0.00003
)

EPOCHS = 20

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for images, labels in loader:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"[F2] Epoch {epoch+1}/{EPOCHS}, "
        f"Loss: {total_loss:.4f}"
    )


# =========================
# GUARDAR
# =========================
torch.save(
    model.state_dict(),
    "../models/modelo_efficientnet.pth"
)

print("Modelo actualizado guardado")

In [ ]:
torch.save(model.state_dict(), "../models/modelo_efficientnet.pth")
print("Modelo guardado correctamente")

In [ ]:
import json

ruta = r"C:\Users\brayn\Desktop\DOCS DEV\Clasificador de Alimentos\models\classes.json"

with open(ruta, "w") as f:
    json.dump(CLASSES, f)